In [2]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# URL for the API (your provided link)

# łączna długość dróg w kilometrach w województwach
url = "https://bdl.stat.gov.pl/api/v1/data/by-variable/77237?format=xml&year=2024&unit-level=2&page=0&page-size=100"

# Download XML data
response = requests.get(url)
response.raise_for_status()  # ensure request was successful
xml_data = response.text

# Parse the XML
root = ET.fromstring(xml_data)

# Extract relevant data
records = []
for unit in root.findall(".//unitData"):
    name = unit.find("name").text
    val = unit.find(".//val").text
    year = unit.find(".//year").text
    records.append({
        "Region": name,
        "Year": int(year),
        "Value": float(val)
    })

# Convert to pandas DataFrame
df = pd.DataFrame(records)

# Display or save
print(df)
#df.to_csv("bdl_data_2024_.csv", index=False)

                 Region  Year    Value
0           MAŁOPOLSKIE  2024  31838.9
1               ŚLĄSKIE  2024  24539.6
2              LUBUSKIE  2024  14896.1
3         WIELKOPOLSKIE  2024  41167.4
4    ZACHODNIOPOMORSKIE  2024  18676.5
5          DOLNOŚLĄSKIE  2024  24632.9
6              OPOLSKIE  2024  10409.5
7    KUJAWSKO-POMORSKIE  2024  27711.9
8             POMORSKIE  2024  20982.9
9   WARMIŃSKO-MAZURSKIE  2024  22304.0
10              ŁÓDZKIE  2024  25482.8
11       ŚWIĘTOKRZYSKIE  2024  17203.6
12            LUBELSKIE  2024  37073.2
13         PODKARPACKIE  2024  20577.7
14            PODLASKIE  2024  27367.3
15          MAZOWIECKIE  2024  56326.3


In [5]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# Lata, które chcesz pobrać
years = [2020, 2021, 2022, 2023, 2024]

# Adres API (bez 'year' na sztywno)
base_url = "https://bdl.stat.gov.pl/api/v1/data/by-variable/77237?format=xml&unit-level=2&page-size=100"

records = []

for year in years:
    url = f"{base_url}&year={year}"
    print(f"Pobieram dane dla roku {year}...")

    response = requests.get(url)
    response.raise_for_status()
    xml_data = response.text

    root = ET.fromstring(xml_data)

    for unit in root.findall(".//unitData"):
        name = unit.find("name").text
        val_el = unit.find(".//val")
        val = float(val_el.text) if val_el is not None and val_el.text else None
        records.append({
            "Region": name,
            "Year": year,
            "Value": val
        })

# Tworzenie DataFrame
df = pd.DataFrame(records)

# Przekształcenie do formatu szerokiego (pivot)
df_wide = df.pivot(index="Region", columns="Year", values="Value").reset_index()

# (Opcjonalnie) zmiana nazw kolumn na czytelniejsze
df_wide.columns.name = None
df_wide.rename(columns={2023: "2023", 2024: "2024"}, inplace=True)

print(df_wide)

# Zapis do pliku
# df_wide.to_csv("bdl_data_2010_2024_wide.csv", index=False)

Pobieram dane dla roku 2020...
Pobieram dane dla roku 2021...
Pobieram dane dla roku 2022...
Pobieram dane dla roku 2023...
Pobieram dane dla roku 2024...
                 Region     2020     2021     2022     2023     2024
0          DOLNOŚLĄSKIE  25322.3  25427.2  25562.7  24873.3  24632.9
1    KUJAWSKO-POMORSKIE  27440.3  27298.0  27911.6  27695.8  27711.9
2             LUBELSKIE  38483.2  38797.0  37695.0  37625.2  37073.2
3              LUBUSKIE  16399.5  15804.4  15778.0  15725.3  14896.1
4           MAZOWIECKIE  55476.7  55810.2  55616.8  56188.2  56326.3
5           MAŁOPOLSKIE  31400.2  31507.5  31870.0  32153.1  31838.9
6              OPOLSKIE  10406.3  10568.1  10487.7  10479.1  10409.5
7          PODKARPACKIE  21630.6  21735.9  21770.2  21899.2  20577.7
8             PODLASKIE  27155.6  27390.7  27452.3  27749.7  27367.3
9             POMORSKIE  23251.2  22781.9  21442.4  21411.1  20982.9
10  WARMIŃSKO-MAZURSKIE  22244.7  22097.6  22306.6  22424.8  22304.0
11        WIELKOP